In [1]:
from datetime import datetime
import random
from faker import Faker
import pandas as pd

In [3]:
# Initialize Faker
fake = Faker()
random.seed(42)  # For reproducible results

# --- 1. CONFIGURATION & BUSINESS LOGIC ---

In [6]:
# Service Centre Weightings: Walvis Bay 40%, Ongwediva 30%, Rundu 30%
SERVICE_CENTRES = ["Walvis Bay", "Ongwediva", "Rundu"]
CENTRE_WEIGHTS = [0.40, 0.30, 0.30]

In [8]:
# Medical Aid Weightings: State 64%, PSEMAS 23%, Private/NMC 13%
MEDICAL_AIDS = ["PRIVATE STATE", "PSEMAS", "NMC / PRIVATE"]
MED_AID_WEIGHTS = [0.64, 0.23, 0.13]

In [10]:
# Procedure Codes & Amounts (Excl. VAT)
# Chronic: ~95% of sessions, Acute: ~5% of sessions
PROCEDURE_ITEMS = {
    "Chronic": [
        {
            "code": "75148",
            "desc": "Chronic haemodialysis (bicarbonate dialysate)",
            "amount": 3160.70,
        },
        {
            "code": "75148",
            "desc": "Chronic haemodialysis (bicarbonate dialysate)",
            "amount": 1935.30,
        },
    ],
    "Acute": [
        {
            "code": "75150",
            "desc": "Acute haemodialysis - initial treatment",
            "amount": 3323.30,
        },
        {
            "code": "75145",
            "desc": "Acute haemodialysis - hospital unit",
            "amount": 2850.00,
        },
        {"code": "75149", "desc": "Acute haemodialysis - ICU", "amount": 4050.00},
    ],
}

In [12]:
# Frequently used Dialysis Stock Consumables (Claim Code 75175 / 0201 / SCRIPT)
STOCK_ITEMS = [
    {
        "code": "75175",
        "desc": "WEBCOL ALCOHOL SWABS CSM WEB",
        "item_code": "0201I6A8",
        "amount": 3.73,
    },
    {
        "code": "75175",
        "desc": "ADMIN SET ADULT CSM 20DRP",
        "item_code": "0201Q228",
        "amount": 30.37,
    },
    {
        "code": "75175",
        "desc": "SODIUM CHLORIDE 0.9% 1000ML",
        "item_code": "0201S100",
        "amount": 17.39,
    },
    {
        "code": "75175",
        "desc": "SYRINGE 20ML LUER LOCK",
        "item_code": "0201SY20",
        "amount": 3.14,
    },
    {
        "code": "75175",
        "desc": "AV FISTULA NEEDLE 16G",
        "item_code": "0201AV16",
        "amount": 18.63,
    },
    {
        "code": "0201",
        "desc": "DIALYZER HIGH FLUX FX80",
        "item_code": "0201FX80",
        "amount": 385.00,
    },
    {
        "code": "SCRIPT",
        "desc": "HEPARIN SODIUM 5000IU/ML INJ",
        "item_code": "GMEDHEP5",
        "amount": 25.88,
    },
    {
        "code": "SCRIPT",
        "desc": "PARACETAMOL 500MG TABS",
        "item_code": "GMEDPARA",
        "amount": 1.25,
    },
]

In [14]:
# Create a realistic recurring patient pool (100 patients)
PATIENT_POOL = []
for i in range(100):
    PATIENT_POOL.append(
        {
            "patient_id": f"PAT-{1000 + i}",
            "patient_name": fake.name(),
            "medical_aid": random.choices(
                MEDICAL_AIDS, weights=MED_AID_WEIGHTS
            )[0],
            "service_centre": random.choices(
                SERVICE_CENTRES, weights=CENTRE_WEIGHTS
            )[0],
        }
    )

# --- 2. GENERATION FUNCTION ---

In [17]:
def generate_monthly_dialysis_data(
    year: int, month: int, num_sessions: int = 150, start_invoice_no: int = 10000
):
    """Generates an end-to-end dialysis clinic dataset for a given month."""
    invoice_rows = []
    current_invoice_no = start_invoice_no

    for _ in range(num_sessions):
        current_invoice_no += 1

        # Select a recurring patient from our pool
        patient = random.choice(PATIENT_POOL)

        # Decide if Chronic (~95%) or Acute (~5%)
        category = random.choices(["Chronic", "Acute"], weights=[0.95, 0.05])[0]

        # Pick Procedure line item
        proc_details = random.choice(PROCEDURE_ITEMS[category])

        # Treatment date within the month
        trans_date = fake.date_time_between_dates(
            datetime(year, month, 1), datetime(year, month, 28)
        ).strftime("%Y-%m-%d")

        # Check for Reversal Logic (~4% chance of being reversed later)
        is_reversal = random.random() < 0.04

        # --- LINE ITEM 1: MAIN PROCEDURE ---
        proc_row = {
            "Invoice_No": current_invoice_no,
            "Trans_Date": trans_date,
            "Service_Centre": patient["service_centre"],
            "Patient_ID": patient["patient_id"],
            "Patient_Name": patient["patient_name"],
            "Medical_Aid": patient["medical_aid"],
            "Category": category,
            "Claim_Code": proc_details["code"],
            "Item_Code": proc_details["code"],
            "Description": proc_details["desc"],
            "Quantity": 1,
            "Amount_Excl": proc_details["amount"],
            "Cost": 0.00,  # Overheads handled separately
            "Is_Reversal": False,
        }
        invoice_rows.append(proc_row)

        # --- LINE ITEMS 2-N: STOCK & CONSUMABLES ---
        # Each dialysis session uses 3 to 6 consumable items
        num_stock_items = random.randint(3, 6)
        session_stock_items = random.sample(STOCK_ITEMS, k=num_stock_items)

        for stock in session_stock_items:
            qty = random.randint(1, 2)
            amount_excl = round(stock["amount"] * qty, 2)

            # Stock direct cost margin: Cost is between 40% and 65% of billed amount
            margin_pct = random.uniform(0.40, 0.65)
            cost = round(amount_excl * margin_pct, 2)

            stock_row = {
                "Invoice_No": current_invoice_no,
                "Trans_Date": trans_date,
                "Service_Centre": patient["service_centre"],
                "Patient_ID": patient["patient_id"],
                "Patient_Name": patient["patient_name"],
                "Medical_Aid": patient["medical_aid"],
                "Category": "Stock",
                "Claim_Code": stock["code"],
                "Item_Code": stock["item_code"],
                "Description": stock["desc"],
                "Quantity": qty,
                "Amount_Excl": amount_excl,
                "Cost": cost,
                "Is_Reversal": False,
            }
            invoice_rows.append(stock_row)

        # --- IF REVERSED: GENERATE NEGATIVE CREDIT NOTE INVOICE ---
        if is_reversal:
            # Reversal happens a few days later
            rev_date = fake.date_time_between_dates(
                datetime.strptime(trans_date, "%Y-%m-%d"),
                datetime(year, month, 28),
            ).strftime("%Y-%m-%d")

            # Duplicate procedure row as negative
            neg_proc = proc_row.copy()
            neg_proc["Trans_Date"] = rev_date
            neg_proc["Quantity"] = -1
            neg_proc["Amount_Excl"] = -abs(proc_row["Amount_Excl"])
            neg_proc["Cost"] = 0.00
            neg_proc["Is_Reversal"] = True
            invoice_rows.append(neg_proc)

            # Duplicate stock rows as negative
            for stock_r in invoice_rows[-num_stock_items - 1 : -1]:
                neg_stock = stock_r.copy()
                neg_stock["Trans_Date"] = rev_date
                neg_stock["Quantity"] = -abs(stock_r["Quantity"])
                neg_stock["Amount_Excl"] = -abs(stock_r["Amount_Excl"])
                neg_stock["Cost"] = -abs(stock_r["Cost"])
                neg_stock["Is_Reversal"] = True
                invoice_rows.append(neg_stock)

    df = pd.DataFrame(invoice_rows)
    return df

In [19]:
# --- 3. RUN SCRIPT & INSPECT ---
if __name__ == "__main__":
    current_time = datetime.now()

    # Generate dataset for current month with 200 dialysis sessions (~1,000 line items)
    df_dialysis = generate_monthly_dialysis_data(
        year=current_time.year,
        month=current_time.month,
        num_sessions=200,
    )

    # Save to local CSV file
    df_dialysis.to_csv("dialysis_transactions.csv", index=False)

    print("--- DATASET SUMMARY ---")
    print(f"Total Rows Generated: {len(df_dialysis)}")
    print("\nCategory Distribution:")
    print(df_dialysis["Category"].value_counts())
    print("\nService Centre Distribution (Sessions):")
    print(
        df_dialysis[df_dialysis["Category"] != "Stock"][
            "Service_Centre"
        ].value_counts(normalize=True)
    )
    print("\nSample Data (First 5 Rows):")
    print(
        df_dialysis[
            [
                "Invoice_No",
                "Trans_Date",
                "Service_Centre",
                "Category",
                "Description",
                "Amount_Excl",
                "Cost",
            ]
        ].head()
    )

--- DATASET SUMMARY ---
Total Rows Generated: 1169

Category Distribution:
Category
Stock      961
Chronic    203
Acute        5
Name: count, dtype: int64

Service Centre Distribution (Sessions):
Service_Centre
Walvis Bay    0.418269
Ongwediva     0.399038
Rundu         0.182692
Name: proportion, dtype: float64

Sample Data (First 5 Rows):
   Invoice_No  Trans_Date Service_Centre Category  \
0       10001  2026-07-03     Walvis Bay  Chronic   
1       10001  2026-07-03     Walvis Bay    Stock   
2       10001  2026-07-03     Walvis Bay    Stock   
3       10001  2026-07-03     Walvis Bay    Stock   
4       10001  2026-07-03     Walvis Bay    Stock   

                                     Description  Amount_Excl   Cost  
0  Chronic haemodialysis (bicarbonate dialysate)      3160.70   0.00  
1                         PARACETAMOL 500MG TABS         1.25   0.81  
2                      ADMIN SET ADULT CSM 20DRP        60.74  38.37  
3                         SYRINGE 20ML LUER LOCK       